# Student Score Prediction

Predicting final-year grades (G3) for secondary school students from demographic, social and
academic features, and testing whether that prediction is still possible *before* first- and
second-term grades are recorded.

Completed as the final project for Machine Learning Fundamentals.

## Problem

The brief for this project sets up the following scenario: the advising team for a large
Portuguese school system wants to predict student performance so it can identify students who
need additional support and direct intervention resources toward them.

That framing raises two questions, and the second is where the interesting work sits:

1. How accurately can a student's final grade (G3) be predicted from the available data?
2. Can it be predicted *without* the first- and second-term grades (G1 and G2)? A model that
   only works once two terms of grades exist is far less useful to an advising team than one
   that flags students early, so both cases are built and compared throughout.

## Load and split the data

The dataset is a 395-student extract of the UCI Student Performance data covering the
Mathematics course, with 35 columns. `G3` — the final-year grade on the Portuguese 0–20 scale —
is the target, and is separated out before the split.

The split is held at `test_size=0.2` and `random_state=42` so the results stay reproducible.

In [36]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.svm import LinearSVR
from sklearn.metrics import root_mean_squared_error, r2_score

In [37]:
student_data = pd.read_csv("data/student-mat.csv")

In [38]:
student_data.head()   # Exploring

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,goout,Dalc,Walc,health,absences_G1,absences_G2,absences_G3,G1,G2,G3
0,GP,F,18.0,U,GT3,A,4,4,at_home,teacher,...,4,1,1,3,1.0,0.0,4.0,5,6,6
1,GP,F,17.0,U,GT3,T,1,1,at_home,other,...,3,1,1,3,NaN,0.0,1.0,5,5,6
2,GP,F,15.0,U,LE3,T,1,1,at_home,other,...,2,2,3,3,1.0,2.0,8.0,7,8,10
3,GP,F,15.0,U,GT3,T,4,2,health,services,...,2,1,1,5,0.0,1.0,2.0,15,14,15
4,GP,F,NaN,U,GT3,T,3,3,other,other,...,2,1,2,5,0.0,0.0,4.0,6,10,10


In [39]:
student_data.info()   # Exploring

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 35 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   school       395 non-null    object 
 1   sex          395 non-null    object 
 2   age          383 non-null    float64
 3   address      395 non-null    object 
 4   famsize      395 non-null    object 
 5   Pstatus      395 non-null    object 
 6   Medu         395 non-null    int64  
 7   Fedu         395 non-null    int64  
 8   Mjob         395 non-null    object 
 9   Fjob         395 non-null    object 
 10  reason       395 non-null    object 
 11  guardian     395 non-null    object 
 12  traveltime   395 non-null    int64  
 13  studytime    395 non-null    int64  
 14  failures     395 non-null    int64  
 15  schoolsup    395 non-null    object 
 16  famsup       395 non-null    object 
 17  paid         395 non-null    object 
 18  activities   395 non-null    object 
 19  nursery 

In [40]:
# Separate target column
target = student_data['G3']
student_data.drop(columns=['G3'], inplace=True)

In [41]:
X_train, X_test, y_train, y_test = train_test_split(student_data, target, test_size=0.2, random_state=42)

## Explore the data

Exploration was done in Tableau rather than matplotlib. The exported charts are in
[`images/`](images/) and the full discussion is in
[`reports/executive-summary.pdf`](reports/executive-summary.pdf).

Four findings shaped the modelling decisions that follow:

- G1 and G2 track G3 almost linearly, and the average of the two tracks it more tightly still.
  This is the signal the model leans on wherever those columns are available.
- 38 of the 395 students have a final grade of 0 despite non-zero first-term grades (G1 between
  4 and 12). They show up as a flat row along the bottom of the grade scatter plots and are the
  largest single source of residual error, since nothing else in the feature set anticipates them.
- Among the non-grade features, intent to pursue higher education separates most clearly:
  average G3 of 10.61 against 6.80.
- Everything else is weak. Travel time to school moves the average from 10.78 down to 8.75
  across its four levels, and going out with friends spans about two points non-monotonically.
  This is the first sign that the without-grades model will struggle.

## Prepare the data

### Feature selection

Twelve of the 35 columns are carried forward. The three `absences_*` columns plus `G1` and `G2`
are needed by the custom transformer below; `failures` and `age` were added on the strength of
the exploration. `higher` is the categorical feature that separated the target most clearly.

`Walc`, `Dalc`, `goout` and `traveltime` are 1–5 survey scales, so they are treated as ordinal
rather than one-hot encoded — the ordering carries meaning that one-hot would discard.

In [43]:
numeric_columns = ["absences_G1", "absences_G2", "absences_G3", "G1", "G2", "failures", "age"]
categorical_columns = ["higher"]
ordinal_columns = ["Walc", "Dalc", "goout", "traveltime"]

### Custom transformer

`FinalProjectTransformer` does two jobs:

- Sums `absences_G1`, `absences_G2` and `absences_G3` into a single `absences_sum` column and
  drops the originals. Total absences across the year is the quantity of interest; keeping the
  three terms separate spends three dimensions on it for no obvious gain.
- Drops `G1` and `G2` when `drop_grades=True`. This flag is what makes the with/without
  comparison cheap: the same transformer definition feeds both variants of the pipeline, so the
  two models differ in exactly one respect and nothing else.

It inherits from `BaseEstimator` and `TransformerMixin` and operates on a DataFrame, which is
why the imputer feeding it is set to `transform="pandas"` output.

In [44]:
class FinalProjectTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, drop_grades=False):
        self.drop_grades = drop_grades

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        absences_sum = X.loc[:, ["absences_G1", "absences_G2", "absences_G3"]].sum(axis=1).rename("absences_sum")
        X = pd.concat([X, absences_sum], axis=1).drop(columns=["absences_G1", "absences_G2", "absences_G3"])
        if self.drop_grades:
            X = X.drop(columns=["G1", "G2"])
        return X

### Pipelines

Three pipelines handle the three column types, combined by a `ColumnTransformer`:

- **Numeric** — median imputation, then the custom transformer, then `StandardScaler`. Built
  twice, once with `drop_grades=False` and once with `True`. Median rather than mean imputation
  because `age`, `absences_G1` and `absences_G2` are the columns with missing values and all
  are skewed.
- **Categorical** — most-frequent imputation, then one-hot encoding with `drop="first"` to
  avoid collinear dummy columns.
- **Ordinal** — most-frequent imputation, then ordinal encoding.

The result is two transformed training sets, `X_train_transformed_with_grades` and
`X_train_transformed_without_grades`, the second with two fewer columns.

In [45]:
# Numerical Pipelines
numeric_pipeline_with_grades = make_pipeline(
    SimpleImputer(strategy="median").set_output(transform="pandas"),
    FinalProjectTransformer(),
    StandardScaler()
)

numeric_pipeline_without_grades = make_pipeline(
    SimpleImputer(strategy="median").set_output(transform="pandas"),
    FinalProjectTransformer(drop_grades=True),
    StandardScaler()
)

In [46]:
# Categorical Pipeline
categorical_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent").set_output(transform="pandas"),
    OneHotEncoder(drop="first")
)

In [47]:
#Ordinal Pipeline
ordinal_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent").set_output(transform="pandas"),
    OrdinalEncoder()    
)

In [48]:
# Column Transformers
column_transformer_with_grades = ColumnTransformer([
    ("numeric", numeric_pipeline_with_grades, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns),
    ("ordinal", ordinal_pipeline, ordinal_columns)
])

column_transformer_without_grades = ColumnTransformer([
    ("numeric", numeric_pipeline_without_grades, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns),
    ("ordinal", ordinal_pipeline, ordinal_columns)
])

In [49]:
# Transform X_train datasets
X_train_transformed_with_grades = column_transformer_with_grades.fit_transform(X_train)
X_train_transformed_without_grades = column_transformer_without_grades.fit_transform(X_train)

## Compare candidate models

Three linear regressors — `LinearRegression`, `LinearSVR` and `Lasso` — are scored with 3-fold
cross-validation against both transformed training sets, giving six results.

The metric is RMSE, which scikit-learn reports as `neg_root_mean_squared_error`, so the printed
values are negative and closer to zero is better.

In [50]:
lin_reg = LinearRegression()
svm_reg = LinearSVR()
lasso_reg = Lasso()

In [51]:
cv_scores_lin_reg_with_grades = cross_val_score(lin_reg, X_train_transformed_with_grades, y_train, 
                                                scoring="neg_root_mean_squared_error", cv=3)

cv_scores_lin_reg_without_grades = cross_val_score(lin_reg, X_train_transformed_without_grades, y_train, 
                                                   scoring="neg_root_mean_squared_error", cv=3)

cv_scores_svm_with_grades = cross_val_score(svm_reg, X_train_transformed_with_grades, y_train, 
                                            scoring="neg_root_mean_squared_error", cv=3)

cv_scores_svm_without_grades = cross_val_score(svm_reg, X_train_transformed_without_grades, y_train, 
                                               scoring="neg_root_mean_squared_error", cv=3)

cv_scores_lasso_with_grades = cross_val_score(lasso_reg, X_train_transformed_with_grades, y_train, 
                                              scoring="neg_root_mean_squared_error", cv=3)

cv_scores_lasso_without_grades = cross_val_score(lasso_reg, X_train_transformed_without_grades, y_train, 
                                                 scoring="neg_root_mean_squared_error", cv=3)

In [52]:
rmse_lin_reg_with_grades = cv_scores_lin_reg_with_grades.mean()
rmse_lin_reg_without_grades = cv_scores_lin_reg_without_grades.mean()
rmse_svm_with_grades = cv_scores_svm_with_grades.mean()
rmse_svm_without_grades = cv_scores_svm_without_grades.mean()
rmse_lasso_with_grades = cv_scores_lasso_with_grades.mean()
rmse_lasso_without_grades = cv_scores_lasso_without_grades.mean()

In [53]:
print(rmse_lin_reg_with_grades)
print(rmse_lin_reg_without_grades)
print(rmse_svm_with_grades)
print(rmse_svm_without_grades)
print(rmse_lasso_with_grades)
print(rmse_lasso_without_grades)

-1.8820480009017893
-4.356848565211963
-2.0069679172731685
-4.3690992062141545
-2.1690514380657917
-4.386455909382055


## Tune the LinearSVR

`LinearSVR` is taken forward to hyperparameter tuning, with `GridSearchCV` searching `C`,
`max_iter` and `tol` against both training sets.

One thing to read carefully below: `GridSearchCV` is left on its default scoring, so the
`best_score_` values printed here are R², not the RMSE reported in the section above.

In [54]:
param_grid = {"tol": [1e-6, 1e-5, 1e-4, 1e-3], 
              "C": [0.0001, 0.001, 0.01, 0.1, 0.25, 0.5, 0.75, 1, 1.25, 1.5, 1.75],
              "max_iter": [1000, 10000, 100000, 1000000]}

In [55]:
grid_search = GridSearchCV(svm_reg, param_grid)

In [56]:
best_params_with_grades = grid_search.fit(X_train_transformed_with_grades, y_train).best_params_
best_params_without_grades = grid_search.fit(X_train_transformed_without_grades, y_train).best_params_

print(best_params_with_grades)
print(best_params_without_grades)

{'C': 1.5, 'max_iter': 100000, 'tol': 1e-06}
{'C': 0.5, 'max_iter': 10000, 'tol': 0.0001}


In [57]:
print(grid_search.fit(X_train_transformed_with_grades, y_train).best_score_)
print(grid_search.fit(X_train_transformed_without_grades, y_train).best_score_)

0.8154944374194288


0.10196892237742813


## Measure performance on the test set

The tuned `LinearSVR` is fitted to both training sets, the held-out test set is transformed
through the same column transformers, and performance is measured with RMSE and R².

The four values printed below are, in order: RMSE with grades, R² with grades, RMSE without
grades, R² without grades.

In [58]:
# Fitting best models
svm_reg_with = LinearSVR(tol=1e-6, C=1.5, max_iter=100000)
svm_reg_without = LinearSVR(tol=1e-4, C=0.5, max_iter=1000)

svm_reg_with_grades = svm_reg_with.fit(X_train_transformed_with_grades, y_train)
svm_reg_without_grades = svm_reg_without.fit(X_train_transformed_without_grades, y_train)

In [59]:
# Transforming test set
X_test_transformed_with_grades = column_transformer_with_grades.transform(X_test)
X_test_transformed_without_grades = column_transformer_without_grades.transform(X_test)

In [60]:
# Evaluting models
y_pred_with = svm_reg_with_grades.predict(X_test_transformed_with_grades)
y_pred_without = svm_reg_without_grades.predict(X_test_transformed_without_grades)

rmse_with_grades = root_mean_squared_error(y_test, y_pred_with)
r2_with_grades = r2_score(y_test, y_pred_with)
rmse_without_grades = root_mean_squared_error(y_test, y_pred_without)
r2_without_grades = r2_score(y_test, y_pred_without)

In [61]:
print(rmse_with_grades)
print(r2_with_grades)
print(rmse_without_grades)
print(r2_without_grades)

2.085720587946679
0.7878458745247777
4.404534923849153
0.053894992691884136


## Results

Cross-validated RMSE on the training set (3-fold):

| Model | With G1/G2 | Without G1/G2 |
| --- | --- | --- |
| Linear Regression | 1.88 | 4.36 |
| LinearSVR (defaults) | 2.01 | 4.37 |
| Lasso | 2.17 | 4.39 |

Held-out test set, tuned `LinearSVR`:

| | RMSE | R² |
| --- | --- | --- |
| With G1/G2 | 2.09 | 0.788 |
| Without G1/G2 | 4.40 | 0.054 |

With the first two terms' grades available, the final grade is predicted to within roughly 2.1
points on a 0–20 scale. Without them the model reaches an R² of 0.054, meaning it reduces
squared error by about 5% against simply predicting the mean grade.

The executive summary recommends using the two models at different intervention tiers: the
with-grades model for higher-cost interventions once first- and second-term grades exist, and
the without-grades model only as a low-cost early flag for Tier 1 advisor check-ins, where a
false positive costs a conversation. That R² of 0.054 is the caveat the recommendation has to
carry — on this feature set, an early flag is only marginally better than picking at random,
and closing that gap needs data the dataset does not contain.

Full discussion is in [`reports/executive-summary.pdf`](reports/executive-summary.pdf).